In [1]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.2 MB/s eta 0:00:00


In [2]:
import os
import json
import shutil
import pandas as pd
import re
from rapidfuzz import process, fuzz
from sklearn.model_selection import train_test_split


# =====================================================
# TARGET LABELS
# =====================================================
LABELS = {
0:'THANKYOU',1:'HELLO',2:'BYE',3:'YES',4:'NO',5:'PLEASE',
6:'SORRY',7:'HELP',8:'WANT1',9:'NEED',10:'HAVE',11:'GO',
12:'EAT1',13:'SLEEP',14:'DRINK1',15:'WHAT1',16:'WHERE',
17:'WHY',18:'COME',19:'YOU',20:'ME',21:'WE',22:'GOOD',
23:'STOP',24:'WAIT'
}

TARGET_GLOSSES = set(LABELS.values())


# =====================================================
# PATHS
# =====================================================
ASL_ROOT = "/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen"
WLASL_ROOT = "/kaggle/input/datasets/risangbaskoro/wlasl-processed"
BAI_ROOT = "/kaggle/input/datasets/yushi97/bai-datasets/bai_dataset"   # <-- NEW DATASET

ASL_SPLITS = f"{ASL_ROOT}/splits"
ASL_VIDEO_DIR = f"{ASL_ROOT}/videos"

WLASL_VIDEO_DIR = f"{WLASL_ROOT}/videos"
WLASL_JSON = f"{WLASL_ROOT}/WLASL_v0.3.json"

OUTPUT = "/kaggle/working/merged_asl"
VIDEO_OUTPUT = f"{OUTPUT}/videos"

os.makedirs(VIDEO_OUTPUT, exist_ok=True)

data = []


# =====================================================
# NORMALIZATION
# =====================================================
def normalize(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)
    return text

normalized_targets = {normalize(g): g for g in TARGET_GLOSSES}


# =====================================================
# FUZZY MATCH FUNCTION
# =====================================================
def match_gloss(word):
    word_norm = normalize(word)

    match, score, _ = process.extractOne(
        word_norm,
        list(normalized_targets.keys()),
        scorer=fuzz.ratio
    )

    if score >= 80:
        return normalized_targets[match]

    return None


# =====================================================
# 1️⃣ PROCESS ASL CITIZEN
# =====================================================
print("\n========== ASL CITIZEN REPORT ==========")

split_files = ["train.csv","test.csv","val.csv"]
asl_counter = {}

for split in split_files:

    path = os.path.join(ASL_SPLITS, split)

    if not os.path.exists(path):
        continue

    print(f"\nProcessing {split}")

    df_split = pd.read_csv(path)

    for _, row in df_split.iterrows():

        gloss = str(row["Gloss"]).upper()

        if gloss not in TARGET_GLOSSES:
            continue

        video_file = row["Video file"]

        unique_name = f"ASL_{video_file}"

        data.append({
            "VideoID": unique_name,
            "Gloss": gloss,
            "Source": "ASL",
            "Path": os.path.join(ASL_VIDEO_DIR, video_file)
        })

        asl_counter[gloss] = asl_counter.get(gloss,0)+1

print("\nASL collected:", sum(asl_counter.values()))


# =====================================================
# 2️⃣ PROCESS WLASL
# =====================================================
print("\n========== WLASL MATCH REPORT ==========")

with open(WLASL_JSON) as f:
    wlasl = json.load(f)

wlasl_counter = {}

for item in wlasl:

    mapped = match_gloss(item["gloss"])

    if mapped is None:
        continue

    for inst in item["instances"]:

        vid = inst["video_id"]

        possible = [
            f"{vid}.mp4",
            f"{vid}.mov",
            f"{vid}.webm"
        ]

        for p in possible:

            full = os.path.join(WLASL_VIDEO_DIR, p)

            if os.path.exists(full):

                unique_name = f"WLASL_{mapped}_{p}"

                data.append({
                    "VideoID": unique_name,
                    "Gloss": mapped,
                    "Source": "WLASL",
                    "Path": full
                })

                wlasl_counter[mapped] = wlasl_counter.get(mapped,0)+1
                break

print("\nWLASL collected:", sum(wlasl_counter.values()))


# =====================================================
# 3️⃣ PROCESS BAI DATASET (NEW)
# =====================================================
print("\n========== BAI DATASET REPORT ==========")

bai_counter = {}

for word_folder in os.listdir(BAI_ROOT):

    folder_path = os.path.join(BAI_ROOT, word_folder)

    if not os.path.isdir(folder_path):
        continue

    mapped = match_gloss(word_folder)

    if mapped is None:
        continue

    for file in os.listdir(folder_path):

        if not file.endswith(".mp4"):
            continue

        full_path = os.path.join(folder_path, file)

        unique_name = f"BAI_{mapped}_{file}"

        data.append({
            "VideoID": unique_name,
            "Gloss": mapped,
            "Source": "BAI",
            "Path": full_path
        })

        bai_counter[mapped] = bai_counter.get(mapped, 0) + 1

print("\nBAI collected:", sum(bai_counter.values()))


# =====================================================
# CREATE DATAFRAME
# =====================================================
df = pd.DataFrame(data)

print("\nTotal collected:", len(df))


# =====================================================
# FINAL DISTRIBUTION
# =====================================================
print("\n========== FINAL DATASET DISTRIBUTION ==========")
print(df["Gloss"].value_counts())

print("\n========== SOURCE DISTRIBUTION ==========")
print(df["Source"].value_counts())


# =====================================================
# REMOVE SMALL CLASSES
# =====================================================
counts = df["Gloss"].value_counts()
valid_classes = counts[counts >= 3].index
df = df[df["Gloss"].isin(valid_classes)]


# =====================================================
# TRAIN / VAL / TEST SPLIT
# =====================================================
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Gloss"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["Gloss"],
    random_state=42
)


# =====================================================
# SAVE CSVs
# =====================================================
os.makedirs(OUTPUT, exist_ok=True)

train_df[["VideoID","Gloss"]].to_csv(f"{OUTPUT}/train.csv", index=False)
valid_df[["VideoID","Gloss"]].to_csv(f"{OUTPUT}/valid.csv", index=False)
test_df[["VideoID","Gloss"]].to_csv(f"{OUTPUT}/test.csv", index=False)

print("\nCSV files saved")


# =====================================================
# COPY VIDEOS
# =====================================================
print("\n========== COPYING VIDEOS ==========")

for _, row in df.iterrows():

    src = row["Path"]
    dst = os.path.join(VIDEO_OUTPUT, row["VideoID"])

    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)


# =====================================================
# ZIP DATASET
# =====================================================
shutil.make_archive("/kaggle/working/Merged_ASL",'zip',OUTPUT)

print("\nZIP CREATED → /kaggle/working/Merged_ASL.zip")


========== ASL CITIZEN REPORT ==========

Processing train.csv

Processing test.csv

Processing val.csv

ASL collected: 778

========== WLASL MATCH REPORT ==========

WLASL collected: 330

========== BAI DATASET REPORT ==========

BAI collected: 1428

Total collected: 2536

========== FINAL DATASET DISTRIBUTION ==========
Gloss
NO          219
WHERE       173
YOU         141
SLEEP       132
YES         128
EAT1        111
SORRY       111
GOOD        104
DRINK1      103
WE          102
WANT1        98
GO           97
HELP         97
WHAT1        97
NEED         94
HAVE         88
HELLO        87
PLEASE       85
WHY          82
WAIT         81
ME           72
COME         71
STOP         64
BYE          61
THANKYOU     38
Name: count, dtype: int64

========== SOURCE DISTRIBUTION ==========
Source
BAI      1428
ASL       778
WLASL     330
Name: count, dtype: int64

CSV files saved

========== COPYING VIDEOS ==========

ZIP CREATED → /kaggle/working/Merged_ASL.zip


In [3]:
# =====================================================
# COUNT TOTAL VIDEOS IN FINAL FOLDER
# =====================================================
print("\n========== FINAL VIDEO COUNT ==========")

video_files = [
    f for f in os.listdir(VIDEO_OUTPUT)
    if f.lower().endswith((".mp4", ".mov", ".webm"))
]

print("Total video files in folder:", len(video_files))


========== FINAL VIDEO COUNT ==========
Total video files in folder: 2536
